# Content Based filtering


In [153]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

articles = pd.read_parquet("./articles.parquet")
history = pd.read_parquet("./validation/history.parquet")
behaviors = pd.read_parquet("./validation/behaviors.parquet")

print(articles.columns)
print(history.columns)
print(behaviors.columns)


Index(['article_id', 'title', 'subtitle', 'last_modified_time', 'premium',
       'body', 'published_time', 'image_ids', 'article_type', 'url',
       'ner_clusters', 'entity_groups', 'topics', 'category', 'subcategory',
       'category_str', 'total_inviews', 'total_pageviews', 'total_read_time',
       'sentiment_score', 'sentiment_label'],
      dtype='object')
Index(['user_id', 'impression_time_fixed', 'scroll_percentage_fixed',
       'article_id_fixed', 'read_time_fixed'],
      dtype='object')
Index(['impression_id', 'article_id', 'impression_time', 'read_time',
       'scroll_percentage', 'device_type', 'article_ids_inview',
       'article_ids_clicked', 'user_id', 'is_sso_user', 'gender', 'postcode',
       'age', 'is_subscriber', 'session_id', 'next_read_time',
       'next_scroll_percentage'],
      dtype='object')


In [154]:
articles["full_text"] = (
    articles["title"].fillna("") + " " +
    articles["subtitle"].fillna("") + " " +
    articles["body"].fillna("")
)


In [155]:
# 3. Vectorisation TF-IDF
vectorizer = TfidfVectorizer(
    max_features=10000,  
    stop_words='english',  
    ngram_range=(1, 2)  
)

tfidf_matrix = vectorizer.fit_transform(articles["full_text"])


In [ ]:
user_profiles = {}

for _, row in history.iterrows():
    user_id = row["user_id"]
    article_ids = row["article_id_fixed"]

    indices = [article_id_to_index.get(aid) for aid in article_ids if aid in article_id_to_index]
    
    if indices:  
        article_vectors = tfidf_matrix[indices]
        profile_vector = article_vectors.mean(axis=0) 
        
        user_profiles[user_id] = profile_vector

print(user_profiles)


{14241: matrix([[0.00105546, 0.00807501, 0.        , ..., 0.00046933, 0.0016407 ,
         0.00110941]]), 20396: matrix([[0.        , 0.00289246, 0.        , ..., 0.        , 0.00490826,
         0.00236706]]), 37953: matrix([[0.        , 0.00432218, 0.        , ..., 0.00127268, 0.00093488,
         0.00130683]]), 38910: matrix([[0.00134435, 0.00775357, 0.        , ..., 0.00036621, 0.00178781,
         0.00139434]]), 39221: matrix([[0.00155272, 0.0115943 , 0.00036302, ..., 0.00016594, 0.00037475,
         0.00161021]]), 44213: matrix([[0.00187054, 0.00692161, 0.        , ..., 0.00035389, 0.00174244,
         0.00080766]]), 48300: matrix([[0.00114841, 0.00528   , 0.        , ..., 0.00064252, 0.00144066,
         0.00075158]]), 52527: matrix([[0.00050142, 0.00216582, 0.        , ..., 0.00183087, 0.00231965,
         0.00520636]]), 53890: matrix([[0.00129509, 0.00847997, 0.00044312, ..., 0.00034306, 0.00134376,
         0.00188354]]), 54469: matrix([[0.00242681, 0.00978189, 0.        , ..

In [157]:
def recommend_articles(user_id, top_k=5, already_read=set()):
    if user_id not in user_profiles:
        return []
    user_vec = user_profiles[user_id]
    sims = cosine_similarity(np.asarray(user_vec), tfidf_matrix).flatten()
    print(sims[:10])

    # Exclure les articles déjà lus
    recommended_indices = np.argsort(sims)[::-1]
    recommendations = []
    for idx in recommended_indices:
        aid = articles.iloc[idx]["article_id"]
        if aid not in already_read:
            recommendations.append(aid)
        if len(recommendations) == top_k:
            break
    return recommendations



In [158]:
def evaluate(user_id, top_k=5):
    user_behaviors = behaviors[behaviors["user_id"] == user_id]
    clicked = set()

    for clicked_list in user_behaviors["article_ids_clicked"].dropna():
        clicked.update(clicked_list)

    already_seen = set()
    for inview_list in user_behaviors["article_ids_inview"].dropna():
        already_seen.update(inview_list)

    recs = recommend_articles(user_id, top_k=top_k, already_read=set())

    hits = [1 if r in clicked else 0 for r in recs]
    print("Hits:", hits)  
    return sum(hits) / top_k if top_k > 0 else 0




In [159]:
example_user = history.iloc[0]["user_id"]
recs = recommend_articles(example_user, top_k=5)
score = evaluate(example_user, top_k=5)

print(f"Articles recommandés pour l'utilisateur {example_user}: {recs}")
print(f"Precision@5 = {score:.2f}")

[0.19632825 0.16322829 0.20243965 0.32251031 0.35512409 0.31725151
 0.12611083 0.29107418 0.27981051 0.18419076]
[0.19632825 0.16322829 0.20243965 0.32251031 0.35512409 0.31725151
 0.12611083 0.29107418 0.27981051 0.18419076]
Hits: [0, 0, 0, 0, 0]
Articles recommandés pour l'utilisateur 14241: [5857800, 7142191, 6173924, 9321454, 6965802]
Precision@5 = 0.00
